# Ingestão de Dados — Camada Bronze

## Objetivo

Obter o dataset Titanic do Kaggle, armazená-lo no Unity Catalog Volume e disponibilizá-lo como tabela Delta na camada Bronze, preservando as colunas originais e mantendo os dados sem tratamento analítico.

## Etapas

1. Instalar a biblioteca KaggleHub e baixar o dataset.
2. Copiar o arquivo CSV para o Volume do Unity Catalog.
3. Carregar os dados com PySpark e inspecionar sua estrutura.
4. Gravar os dados na tabela `projeto_titanic.bronze.titanic`.
5. Validar a quantidade de registros, a unicidade do identificador e a presença de valores nulos.

## Resultado esperado

Disponibilizar os dados brutos na camada Bronze para posterior tratamento na Silver, mantendo os valores ausentes existentes na fonte original.

In [0]:
%pip install kagglehub

In [0]:
import kagglehub
import os
import shutil

# Download da versão utilizada no projeto
path = kagglehub.dataset_download("yasserh/titanic-dataset/versions/1")

print("Path dos arquivos baixados:", path)
print("Arquivos baixados:", os.listdir(path))

In [0]:
# Caminho do arquivo baixado pelo kagglehub
arquivo_origem = os.path.join(path, "Titanic-Dataset.csv")

# Caminho do Volume do Unity Catalog
arquivo_volume = "/Volumes/projeto_titanic/bronze/raw_data/Titanic-Dataset.csv"

# Copia o arquivo para o Volume
shutil.copy(arquivo_origem, arquivo_volume)

print("Arquivo copiado para:", arquivo_volume)

In [0]:
#Leitura do arquivo com o PySpark

df_bronze_titanic = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivo_volume)
)

In [0]:
#Visualização do DataFrame criado com o PySpark
display(df_bronze_titanic.limit(5))

In [0]:
#Visualização das colunas e tipos do DataFrame
df_bronze_titanic.printSchema()

In [0]:
print(f"Quantidade de registros: {df_bronze_titanic.count()}")
print(f"Quantidade de colunas: {len(df_bronze_titanic.columns)}")

In [0]:
# Gravação dos dados brutos na camada Bronze como tabela Delta, mantendo a estrutura original do dataset para uso nas etapas seguintes.
(
    df_bronze_titanic.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.bronze.titanic")
)
print("Tabela Delta gravada na camada Bronze com sucesso.")

In [0]:
%sql
-- Validação da integridade dos dados após a ingestão na camada Bronze,
-- verificando volume de registros, unicidade da chave e presença de valores nulos.

SELECT
    'Total de registros' AS validacao,
    CAST(COUNT(*) AS STRING) AS resultado
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Passageiros distintos',
    CAST(COUNT(DISTINCT PassengerId) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Registros duplicados',
    CAST(COALESCE(SUM(quantidade - 1), 0) AS STRING)
FROM (
    SELECT
        PassengerId,
        COUNT(*) AS quantidade
    FROM projeto_titanic.bronze.titanic
    WHERE PassengerId IS NOT NULL
    GROUP BY PassengerId
    HAVING COUNT(*) > 1
) AS ids_duplicados

UNION ALL

SELECT
    'Nulos em PassengerId',
    CAST(SUM(CASE WHEN PassengerId IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Survived',
    CAST(SUM(CASE WHEN Survived IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Pclass',
    CAST(SUM(CASE WHEN Pclass IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Name',
    CAST(SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Sex',
    CAST(SUM(CASE WHEN Sex IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Age',
    CAST(SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Ticket',
    CAST(SUM(CASE WHEN Ticket IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Fare',
    CAST(SUM(CASE WHEN Fare IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Cabin',
    CAST(SUM(CASE WHEN Cabin IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic

UNION ALL

SELECT
    'Nulos em Embarked',
    CAST(SUM(CASE WHEN Embarked IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM projeto_titanic.bronze.titanic;